# FlashEats — Class 7 Challenge
## Model the Business Workflow with Data

### Client question
> **“Show us where in the workflow delay accumulates, how customers react, what interventions we make, and which metrics we should use to improve the project KPI.”**

Do not repeat source discovery, retrieval, or data-cleaning work. Today the goal is to build a useful business-workflow model.

In [1]:
import json, sqlite3, zipfile
from pathlib import Path
import pandas as pd
pd.set_option("display.max_columns",100)
pd.set_option("display.max_colwidth",140)

def find_pack_root(search_root=Path("/content")):
    for candidate in search_root.rglob("FlashEats_Classroom_Pack_V2"):
        if (candidate/"database"/"flasheats.db").exists(): return candidate
    return None

# Local run: the notebook usually already lives inside the pack root.
BASE = None
if (Path.cwd() / "database" / "flasheats.db").exists():
    BASE = Path.cwd()
elif Path.cwd().name == "FlashEats_Classroom_Pack_V2":
    BASE = Path.cwd()

if BASE is None:
    BASE=find_pack_root()
if BASE is None:
    try:
        from google.colab import files
        print("Upload the FlashEats Class 7 classroom pack ZIP.")
        uploaded=files.upload(); zip_name=next(n for n in uploaded if n.endswith(".zip"))
        extract_dir=Path("/content/flasheats_class7"); extract_dir.mkdir(parents=True,exist_ok=True)
        with zipfile.ZipFile(zip_name) as z: z.extractall(extract_dir)
        BASE=find_pack_root(Path("/content"))
    except Exception as e: print(e)
if BASE is None:
    BASE=find_pack_root(Path.cwd())
if BASE is None: raise FileNotFoundError("Could not locate FlashEats_Classroom_Pack_V2")
print("Using pack:",BASE)

Using pack: /Users/parthdagia/flasheats-classroom-pack/divvyRebalancingWithMyIQ/flasheats-challenges


In [2]:
con=sqlite3.connect(BASE/"database"/"flasheats.db")
orders=pd.read_sql("SELECT * FROM orders",con)
customers=pd.read_sql("SELECT * FROM customers",con)
restaurants=pd.read_sql("SELECT * FROM restaurants",con)
drivers=pd.read_sql("SELECT * FROM drivers",con)
tickets=pd.read_csv(BASE/"data"/"support_tickets.csv")
customer_actions=pd.read_csv(BASE/"data"/"customer_app_actions.csv")
interventions=pd.read_csv(BASE/"data"/"order_interventions.csv")
outcomes=pd.read_csv(BASE/"data"/"order_outcomes.csv")
with open(BASE/"data"/"class7_model_brief.json") as f: model_brief=json.load(f)
print("orders",orders.shape,"actions",customer_actions.shape,"interventions",interventions.shape,"outcomes",outcomes.shape)
print("Project KPI:",model_brief["project_kpi"])

orders (1603, 13) actions (2365, 6) interventions (430, 6) outcomes (1600, 6)
Project KPI: Reduce late delivery rate


# Challenge 1 — Reconstruct the order lifecycle

Choose 3 orders:
- one delivered on time,
- one delivered late,
- one with an intervention.

Build a timeline with:

`event_time | event_type | actor | source_system`

Include as many lifecycle events as the data supports.

### Hint
Start from one `order_id`, collect events from each source, then sort by time.

In [3]:
LATE_THRESHOLD_MIN = 10   # our KPI definition from Classes 5 and 6 (VP Ops to confirm)

base_orders = orders.drop_duplicates("order_id", keep="first").copy()

# order_outcomes.late_flag uses ANY delay > 0 (843 "late" = the 56% rule).
# We keep it for comparison but select examples with our > 10 min definition.
outcomes["late_10"] = outcomes["delay_min"] > LATE_THRESHOLD_MIN
print("late_flag (any delay):", int(outcomes["late_flag"].sum()),
      "| late_10 (> 10 min):", int(outcomes["late_10"].sum()))

ts = lambda s: pd.to_datetime(s, format="mixed", errors="coerce")
driver_events = pd.DataFrame([{"driver_id": d["driver_id"], **e}
                              for d in json.load(open(BASE / "data" / "driver_events.json")) for e in d["events"]])
dispatch = pd.DataFrame([r for p in sorted((BASE / "student_output" / "raw_dispatch").glob("page_*.json"))
                         for r in json.loads(p.read_text())["data"]])     # raw pages preserved in Class 5
restaurant_status = pd.read_csv(BASE / "data" / "restaurant_status.csv").drop_duplicates()

def build_order_timeline(order_id):
    o = base_orders.loc[base_orders["order_id"] == order_id].iloc[0]
    rows = [
        (o["created_at"], "ORDER_CREATED", f"customer {o['customer_id']}", "orders (SQLite)"),
        (o["promised_eta"], "ETA_PROMISED (target)", "system", "orders (SQLite)"),
        (o["pickup_at"], "PICKED_UP", f"driver {o['driver_id']}", "orders (SQLite)"),
        (o["actual_delivery_at"], "DELIVERED", f"driver {o['driver_id']}", "orders (SQLite)"),
    ]
    d = dispatch.loc[dispatch["order_id"] == order_id]
    for _, r in d.iterrows():
        rows.append((r["assigned_at"], "DRIVER_ASSIGNED", f"driver {r['original_driver_id']}", "dispatch API"))
        rows.append((r["estimated_pickup_at"], "PICKUP_ESTIMATED (target)", "dispatch", "dispatch API"))
        if pd.notna(r["reassigned_at"]):
            rows.append((r["reassigned_at"], "DRIVER_REASSIGNED", f"driver {r['driver_id']}", "dispatch API"))
    for _, r in restaurant_status.loc[restaurant_status["order_id"] == order_id].iterrows():
        rows.append((r["last_updated_at"], f"RESTAURANT_STATUS: {r['status'].strip().lower()}", f"restaurant {r['restaurant_id']}", "restaurant_status.csv"))
    for _, r in customer_actions.loc[customer_actions["order_id"] == order_id].iterrows():
        rows.append((r["action_at"], r["action_type"], f"customer {r['customer_id']}", "customer_app_actions.csv"))
    for _, r in tickets.loc[tickets["order_id"] == order_id].drop_duplicates("ticket_id").iterrows():
        rows.append((r["created_at"], f"SUPPORT_TICKET: {r['category']}", "customer", "support_tickets.csv"))
    for _, r in interventions.loc[interventions["order_id"] == order_id].iterrows():
        rows.append((r["intervention_at"], f"INTERVENTION: {r['intervention_type']} ({r['reason']})", r["initiated_by"], "order_interventions.csv"))
    pings = driver_events[(driver_events["order_id"] == order_id) & (driver_events["type"] == "gps_ping")]
    for _, r in pings.iterrows():
        rows.append((r["timestamp"], "GPS_PING", f"driver {r['driver_id']}", "driver_events.json"))
    tl = pd.DataFrame(rows, columns=["event_time", "event_type", "actor", "source_system"])
    tl["event_time"] = ts(tl["event_time"])
    tl = tl.dropna(subset=["event_time"]).sort_values("event_time").reset_index(drop=True)
    tl["min_since_order"] = ((tl["event_time"] - tl["event_time"].min()).dt.total_seconds() / 60).round(1)
    return tl

on_time_order = outcomes.loc[outcomes["outcome_bucket"].eq("delivered_on_time"), "order_id"].iloc[0]
late_order = outcomes.loc[outcomes["late_10"] & ~outcomes["order_id"].isin(interventions["order_id"]), "order_id"].iloc[0]
# an intervention order that is also late, to see whether the intervention came in time
iv_late = interventions.merge(outcomes, on="order_id")
intervention_order = iv_late.loc[iv_late["late_10"] & (iv_late["intervention_type"] != "CUSTOMER_CREDIT"), "order_id"].iloc[0]
print("on-time", on_time_order, "| late", late_order, "| intervention", intervention_order)

for label, oid in [("ON TIME", on_time_order), ("LATE", late_order), ("INTERVENTION + LATE", intervention_order)]:
    delay = outcomes.set_index("order_id").loc[oid, "delay_min"]
    print(f"\n=== {label}: {oid} (delay vs ETA: {delay:+.1f} min)")
    display(build_order_timeline(oid))

# Cross-source check found while building timelines: does dispatch confirm reassignment interventions?
reassign_iv = interventions.loc[interventions["intervention_type"] == "DRIVER_REASSIGNMENT", "order_id"]
dispatch_reassigned = dispatch.loc[dispatch["reassigned_at"].notna(), "order_id"]
print(f"DRIVER_REASSIGNMENT interventions: {len(reassign_iv)} | dispatch reassigned_at set: {len(dispatch_reassigned)} "
      f"| in both: {reassign_iv.isin(dispatch_reassigned).sum()}")

late_flag (any delay): 843 | late_10 (> 10 min): 349
on-time O00003 | late O00005 | intervention O00781

=== ON TIME: O00003 (delay vs ETA: -4.4 min)


,event_time,event_type,actor,source_system,min_since_order
0,2026-08-01 19:35:00.000000,ORDER_CREATED,customer C0855,orders (SQLite),0.0
1,2026-08-01 19:35:00.002443,DRIVER_ASSIGNED,driver D039,dispatch API,0.0
2,2026-08-01 19:44:37.017130,GPS_PING,driver D039,driver_events.json,9.6
3,2026-08-01 19:48:00.000000,RESTAURANT_STATUS: preparing,restaurant R010,restaurant_status.csv,13.0
4,2026-08-01 19:53:00.000000,PICKUP_ESTIMATED (target),dispatch,dispatch API,18.0
5,2026-08-01 19:54:14.031817,GPS_PING,driver D039,driver_events.json,19.2
6,2026-08-01 19:57:50.820366,PICKED_UP,driver D039,orders (SQLite),22.8
7,2026-08-01 20:03:51.046504,GPS_PING,driver D039,driver_events.json,28.9
8,2026-08-01 20:13:28.061191,DELIVERED,driver D039,orders (SQLite),38.5
9,2026-08-01 20:17:52.941967,ETA_PROMISED (target),system,orders (SQLite),42.9



=== LATE: O00005 (delay vs ETA: +21.0 min)


,event_time,event_type,actor,source_system,min_since_order
0,2026-08-06 20:35:00.000000,ORDER_CREATED,customer C0040,orders (SQLite),0.0
1,2026-08-06 20:36:55.346822,DRIVER_ASSIGNED,driver D047,dispatch API,1.9
2,2026-08-06 20:53:00.000000,PICKUP_ESTIMATED (target),dispatch,dispatch API,18.0
3,2026-08-06 21:09:53.328119,GPS_PING,driver D047,driver_events.json,34.9
4,2026-08-06 21:18:38.024280,PICKED_UP,driver D047,orders (SQLite),43.6
5,2026-08-06 21:42:51.309416,GPS_PING,driver D047,driver_events.json,67.9
6,2026-08-06 21:54:48.000000,ETA_PROMISED (target),system,orders (SQLite),79.8
7,2026-08-06 22:15:49.290713,DELIVERED,driver D047,orders (SQLite),100.8



=== INTERVENTION + LATE: O00781 (delay vs ETA: +13.1 min)


,event_time,event_type,actor,source_system,min_since_order
0,2026-08-22 12:33:00.000000,ORDER_CREATED,customer C0103,orders (SQLite),0.0
1,2026-08-22 12:40:42.931820,DRIVER_ASSIGNED,driver D103,dispatch API,7.7
2,2026-08-22 12:51:00.000000,PICKUP_ESTIMATED (target),dispatch,dispatch API,18.0
3,2026-08-22 12:54:00.000000,ETA_VIEWED,customer C0103,customer_app_actions.csv,21.0
4,2026-08-22 12:57:00.000000,INTERVENTION: PRIORITY_DISPATCH (late_risk),operations,order_interventions.csv,24.0
5,2026-08-22 13:04:00.000000,RESTAURANT_STATUS: ready,restaurant R019,restaurant_status.csv,31.0
6,2026-08-22 13:07:42.051142,GPS_PING,driver D103,driver_events.json,34.7
7,2026-08-22 13:11:17.309762,PICKED_UP,driver D103,orders (SQLite),38.3
8,2026-08-22 13:22:41.170463,GPS_PING,driver D103,driver_events.json,49.7
9,2026-08-22 13:48:36.000000,ETA_PROMISED (target),system,orders (SQLite),75.6


DRIVER_REASSIGNMENT interventions: 155 | dispatch reassigned_at set: 95 | in both: 8


### Challenge 1 answer

We merged **7 sources** into one `event_time | event_type | actor | source_system` timeline per order. Targets (promised ETA, estimated pickup) are shown as events so each order can be read against its promise.

| Order | Outcome | What the timeline shows |
|:-|:-|:-|
| **O00003** on time | 4.4 min early | Driver assigned instantly, picked up **4.8 min after** the dispatch pickup estimate, 16 min ride |
| **O00005** late | **21 min late** | Picked up **25.6 min after** the pickup estimate (43.6 min after ordering). The time is lost *before* pickup, with nothing in the data to say why: no restaurant status, no customer action, no intervention |
| **O00781** intervention | 13 min late | Customer checks the ETA at 21 min, operations sends a `PRIORITY_DISPATCH` at 24 min, restaurant marks "ready" at 31 min, pickup at 38 min. The intervention came early but the order was still late |

**What the timelines taught us about the data**
1. **The dispatch pickup estimate is always exactly 18 min after ordering**, so it is a fixed rule, not a real estimate.
2. **The gap between "pickup estimated" and "picked up" is where late orders visibly lose time**, which confirms the Class 5 finding at the level of individual orders.
3. **Intervention sources disagree.** Order O00001 has a `DRIVER_REASSIGNMENT` intervention, yet dispatch shows the same driver throughout. Across all orders, only **8 of 155** reassignment interventions are confirmed by dispatch's `reassigned_at` (95 set). An intervention log that does not match the system it claims to have changed cannot be used to evaluate interventions without owner clarification.
4. `order_outcomes.late_flag` uses the **any-delay rule** (843 late = 56%). We keep it but model with `late_10` (> 10 min, 349 late) to stay consistent with Classes 5 and 6.

# Challenge 2 — Define the canonical project model

Your model must support:
1. customer → orders
2. order → customer interactions
3. order → support interactions
4. order → interventions
5. order → outcome

For each table, document:
- primary key,
- important foreign keys,
- grain.

Then explain why this model is better for the project than mirroring every source-system table.

In [4]:
sources={"orders":orders,"customer_actions":customer_actions,"support_tickets":tickets,"interventions":interventions,"outcomes":outcomes}
for name,df in sources.items():
    print(name,df.shape); display(df.head(2))
# TODO: document grain + relationships

orders (1603, 13)


,order_id,customer_id,restaurant_id,driver_id,city,created_at,promised_eta,pickup_at,actual_delivery_at,final_status,distance_km_estimate,traffic_bucket,weather_bucket
0,O00001,C0168,R009,D103,Bengaluru,2026-08-13T12:56:00,2026-08-13T14:11:36,2026-08-13T13:35:49.825561,2026-08-13T14:22:25.035899,delivered,18.00,medium,clear
1,O00002,C0043,R018,D083,Bengaluru,2026-08-01T18:36:00,2026-08-01T19:43:16.627988,2026-08-01T18:58:24.735336,2026-08-01T19:47:14.818533,delivered,9.37,severe,clear


customer_actions (2365, 6)


,action_id,order_id,customer_id,action_type,action_at,channel
0,ACT-00001,O01044,C0249,ETA_VIEWED,2026-08-05T22:11:00,mobile_app
1,ACT-00002,O01334,C0692,ETA_VIEWED,2026-08-01T17:48:00,mobile_app


support_tickets (202, 5)


,ticket_id,order_id,created_at,category,customer_message
0,T00001,NaN,2026-08-26T23:34:00,late_delivery,My order is already past the promised time.
1,T00002,NaN,2026-08-24T13:49:00,eta_changed,The ETA keeps changing and the food is still not here.


interventions (430, 6)


,intervention_id,order_id,intervention_type,intervention_at,initiated_by,reason
0,INT-00001,O00781,PRIORITY_DISPATCH,2026-08-22T12:57:00,operations,late_risk
1,INT-00002,O01476,CUSTOMER_CREDIT,2026-08-01T23:46:36.200640,support,support_resolution


outcomes (1600, 7)


,order_id,final_status_norm,delivered_flag,late_flag,delay_min,outcome_bucket,late_10
0,O00001,delivered,1,1.0,10.82,delivered_late,True
1,O00002,delivered,1,1.0,3.97,delivered_late,False


# Challenge 3 — Build interaction → intervention → outcome

Create one order-level table containing:

`order_id, customer_id, support_opened, cancel_attempted, intervention_count, intervention_types, final_status, late_flag, delay_min`

Answer:
1. How many late orders had support interaction?
2. How many orders received intervention?
3. Which intervention is most common?
4. Which frustrated journeys had no intervention?

### Hint
Aggregate one-to-many tables before joining them to order-level outcomes.

In [5]:
actions_by_order=(customer_actions.groupby("order_id").agg(action_count=("action_id","count")).reset_index())
# TODO: add flags, aggregate interventions, and join to outcomes/orders

# Challenge 4 — Select 3–5 business metrics

Project KPI: **Reduce Late Delivery Rate**.

For each chosen metric, document:
- metric name,
- formula,
- grain,
- why it matters,
- relationship to the project KPI.

At least one metric must represent:
- an outcome,
- a customer interaction,
- an intervention.

In [6]:
# Example starting point:
# valid_outcomes=outcomes[outcomes.late_flag.notna()]
# late_delivery_rate=valid_outcomes.late_flag.mean()

# TODO: calculate your selected metrics

# Challenge 5 — Investigate the workflow with joins and aggregations

Answer at least three:

A. Do orders with support interactions have higher delay?  
B. What is late rate with vs without intervention?  
C. Which intervention type is associated with the lowest late rate?  
D. Which restaurants contribute the largest number of late orders?  
E. Which journeys show support interaction + intervention + still late?

For every answer, add one sentence:

> **What does this tell the business, and what does it NOT prove?**

In [7]:
# TODO: use the order-level model plus joins/groupby
# Reminder: association != causation

# Challenge 6 — Connect the model to the KPI

Create a one-page project view:

`PROJECT KPI → OUTCOME METRIC → WORKFLOW/DRIVER METRICS → INTERVENTIONS → DATA SOURCES/EVENTS`

Answer:
1. Which metrics are directly controllable by operations?
2. Which are outcomes?
3. Which missing event limits the model most?
4. What would you instrument next?

Final deliverable:
- core entities,
- key events,
- relationships,
- 3–5 metrics,
- KPI linkage,
- one modelling limitation.

# Final reflection

A strong solution does not produce the largest schema.

It produces the **smallest useful model that explains the workflow and supports the project KPI**.